# Lab 1 — Data Ingestion: Connect Your Data Sources

### Databricks Data Camp for Business Users 🛍️

**Story so far:** As the new CEO of Databricks Retail Corp., you've set up your
workspace (Lab 0). Your sales data currently lives in an **operational database**
— a **Lakebase** (Databricks-managed Postgres) instance that powers the online
store. On top of that, your strategy team has handed you a **PDF of external
market research**.

To become data-driven, you first need to **get all of this into Databricks**.

In this lab you will:
1. Understand the **two kinds of sources** you're ingesting (a database + files).
2. Use **Lakeflow Connect** to ingest the Lakebase Postgres tables into Unity Catalog.
3. Bring the **market-research PDF** into Databricks.
4. Confirm everything landed in your **bronze** layer.

⏱️ *Estimated time: 25–30 minutes.*

## 1.0 — Load config

Run this first in every lab. It reconnects you to the shared catalog/schema names
from `setup/config.py`.

In [ ]:
import os, sys
def find_repo_root():
    p = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "setup")) and os.path.isdir(os.path.join(p, "labs")):
            return p
        p = os.path.dirname(p)
    return os.getcwd()
REPO_ROOT = find_repo_root()
sys.path.insert(0, os.path.join(REPO_ROOT, "setup"))
from config import get_config
cfg = get_config()
spark.sql(f"USE CATALOG {cfg['CATALOG']}")
print("Catalog:", cfg["CATALOG"], "| Bronze:", cfg["CATALOG_BRONZE"])
print("Lakebase instance:", cfg["LAKEBASE_INSTANCE"])

## 1.1 — What is Lakeflow Connect? (plain-English version)

Think of your data as living in different "buildings":

- 🏢 **The operational database (Lakebase Postgres)** — this is where your live
  store writes every order and customer. It's great for *running* the store, but
  not for *analyzing* it.
- 📄 **Files** — spreadsheets, and in our case a **PDF** of market research.

**Lakeflow Connect** is Databricks' built-in way to **bring data from those other
buildings into your lakehouse** — reliably and on a schedule — without you writing
plumbing code. You point it at a source, pick the tables, and it lands them in
Unity Catalog (your **bronze** layer = the raw, faithful copy).

> **Medallion architecture preview:** we land raw data in **bronze**, clean/join it
> into **silver** (Lab 3), then build business-ready **gold** tables (Lab 3). Bronze
> = "as received."

In **Lab 0** you provisioned and seeded the **Lakebase** source, and you confirmed
your **bronze schema is empty**. This lab is where you fill it — using the *proper*
ingestion path you'd use in real life, rather than pre-loading.

## 1.2 — Ingest with Lakeflow Connect (guided UI steps)

This is the part you do **in the Databricks UI** — it's the click-through a
business user would actually use. Follow along:

1. In the left sidebar, click **Data Ingestion** (or **+ New → Add data**).
2. Choose **Lakeflow Connect**, then pick the **PostgreSQL / Lakebase** source.
3. **Connection:**
   - Select your existing instance **`retail-corp-lakebase`**.
   - Database: **`retail_corp`**.
   - Authentication is handled by Databricks (OAuth) — you don't manage passwords.
4. **Select the tables to ingest** (choose all six):
   - `dim_product`
   - `dim_customer`
   - `fact_orders`
   - `fact_order_items`
   - `fact_marketing_campaigns`
   - `fact_sales_forecast`  *(the Data Science team's forecast hand-off — you'll
     put it to the test in Lab 5)*
5. **Destination:**
   - Catalog: **`retail_corp`**
   - Schema: **`bronze`**
6. **Schedule:** choose **Manual / Triggered** for the lab (in production you'd
   pick a cadence, e.g. hourly). Name the ingestion pipeline
   `retail_corp_bronze_ingest`.
7. Click **Create** and then **Run**.

> 🧭 **Don't see the PostgreSQL connector?** Lakeflow Connect connectors are
> enabled per workspace. If it's missing, that's an admin toggle — ask your
> workspace admin to enable the Lakeflow Connect PostgreSQL/Lakebase connector,
> then come back to this step.

When the pipeline finishes, it will have written the six tables into
`retail_corp.bronze` (which started empty after Lab 0). The next cells verify that.

## 1.3 — (Optional, technical) How the ingestion works under the hood

Prefer to *see* the mechanics behind the UI? Conceptually, Lakeflow Connect reads
each table from the Lakebase Postgres source and writes it into your `bronze`
schema as a Delta table — then keeps it in sync on the schedule you chose. **Skip
this cell if the UI path worked**; it's here for the curious.

In [ ]:
# Informational — describes what the Lakeflow Connect pipeline (1.2) does for you.
for t in cfg["BRONZE_TABLES"]:
    print(f"Lakeflow Connect: Lakebase.{t}  →  {cfg['CATALOG_BRONZE']}.{t}")
print("\nℹ️  Lakeflow Connect manages this end to end — initial load, schema drift,")
print("   and incremental refresh — so you don't write or maintain ingestion code.")

## 1.4 — Verify the bronze layer

Once the Lakeflow Connect pipeline (1.2) has run, your previously-empty bronze
schema should now hold six populated tables. Let's confirm.

In [ ]:
from pyspark.sql import functions as F

rows = []
for t in cfg["BRONZE_TABLES"]:
    fqn = f"{cfg['CATALOG_BRONZE']}.{t}"
    try:
        rows.append((t, spark.table(fqn).count()))
    except Exception as e:
        rows.append((t, f"MISSING — {str(e)[:60]}"))

print(f"{'bronze table':<30}{'rows':>12}")
print("-" * 42)
for t, c in rows:
    print(f"{t:<30}{c:>12,}" if isinstance(c, int) else f"{t:<30}{c:>12}")

In [ ]:
# Quick business peek: revenue by category straight from bronze.
# (We haven't built clean silver/gold yet — this is a rough look.)
display(spark.sql(f'''
    SELECT p.category,
           ROUND(SUM(oi.unit_price * oi.quantity), 0) AS gross_revenue_usd,
           SUM(oi.quantity)                            AS units_sold
    FROM {cfg['CATALOG_BRONZE']}.fact_order_items oi
    JOIN {cfg['CATALOG_BRONZE']}.dim_product p ON oi.product_id = p.product_id
    JOIN {cfg['CATALOG_BRONZE']}.fact_orders o ON oi.order_id = o.order_id
    WHERE o.order_status <> 'CANCELLED'
    GROUP BY p.category
    ORDER BY gross_revenue_usd DESC
'''))

## 1.5 — Bring in the market-research PDF

Your last source is the **external market-research PDF**. There are two ways it
reaches Databricks; you'll use whichever fits:

Let's upload the file to your Volume (from Lab 0's deploy).** Verify with the cell below. For now, just confirm the file is available.

In [ ]:
pdf_dir = f"{cfg['VOLUME_PATH']}/market_research"
try:
    files = dbutils.fs.ls(pdf_dir)
    print("Files in", pdf_dir, ":")
    for f in files:
        print("  •", f.name, f"({f.size:,} bytes)")
    print("\n✓ The market-research PDF is in your Volume. In Lab 5 you'll upload")
    print("  it (from your laptop copy) into a Genie Space.")
except Exception as e:
    print("PDF not found in the Volume:", str(e)[:150])
    print("→ That's OK. You downloaded it from assets/market_research/ in Lab 0.")
    print("  You'll upload it into a Genie Space in Lab 5.")

## ✅ Lab 1 complete

You've connected your data sources:
- ✅ Understood **Lakeflow Connect** and when to use it
- ✅ Ingested the Lakebase Postgres tables into **bronze** with Lakeflow Connect
- ✅ Verified all six bronze tables went from empty to populated
- ✅ Located the market-research **PDF** for Lab 5
- ✅ Took a first rough peek at revenue by category

**Next up → Lab 2: Data Discovery.** You'll use Unity Catalog, Genie One, and
Domains to explore what data you actually have — and fix any missing metadata.